### Задача об упаковке в контейнеры

Задан размер базового контейнера и предметы. Надо упаковать их в контейнеры так, чтобы использовать их минимальное число.

### Online-алгоритмы

На нас движется поток контейнеров и нет ни единой возможности, чтобы принять скольк бы то ни было "стратегическое решение", чистая тактика.

### Алгоритм NF (Следующий подходящий)

В произвольном порядке упаковываем предметы по следующему правилу.  
Первый предмет помещаем в первый контейнер.  
На $k$-м шаге пытаемся поместить $k$-й предмет в текущий контейнер.  
Если предмет входит, то помещаем его и переходим к следующему шагу, иначе помещаем предмет в новый контейнер.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

current_dir = Path.cwd().resolve()
FILE_PATH = current_dir / "data" / "binpack.txt"

print(FILE_PATH.exists())

def read_binpack_file(filepath):
    with open(filepath, 'r') as f:
        lines = f.read().strip().split('\n')
    
    first_line = lines[0]
    capacity = float(first_line.split('=')[1].strip())
    
    weights = [float(line.strip()) for line in lines[1:] if line.strip()]
    
    return capacity, weights

True


In [2]:
def next_fit(weights, capacity):
    bins_used = 0             
    current_remaining = 0.0    
    
    for w in weights:
        # if w <= current_remaining + 1e-9:
        if w <= current_remaining:
            current_remaining -= w
        else:
            bins_used += 1
            current_remaining = capacity - w
            
    return bins_used

In [3]:
capacity, weights = read_binpack_file(FILE_PATH)

print(f"NF")
print(f"Вместимость контейнера: {capacity}")
print(f"Количество предметов: {len(weights)}")
print(f"Суммарный вес: {sum(weights)}")
    
nf_bins = next_fit(weights, capacity)
print(f"Результат NF (Next Fit): {nf_bins} контейнеров")

NF
Вместимость контейнера: 100.0
Количество предметов: 501
Суммарный вес: 16700.0
Результат NF (Next Fit): 186 контейнеров


### Алгоритм FF (Первый подходящий)

В произвольном порядке упаковываем предметы по следующему правилу.  
Первый предмет помещаем в первый контейнер.  

На $k$-м шаге находим контейнер с наименьшим номером, куда помещается $k$-й предмет, и помещаем его туда. Если такого контейнера нет, то берем новый пустой контейнер и помещаем предмет в него.

In [4]:
def first_fit(weights, capacity):
    remaining_spaces = []
    
    for w in weights:
        placed = False
        
        for i in range(len(remaining_spaces)):
            # if remaining_spaces[i] >= w - 1e-9:
            if remaining_spaces[i] >= w:
                remaining_spaces[i] -= w
                placed = True
                break # нашли первый подходящий значит хватит тратить время на поиск

        # ну что поделать... достаем пустой контейнер
        if not placed:
            remaining_spaces.append(capacity - w)
    return len(remaining_spaces)

In [5]:
print(f"FF")    
    
ff_bins = first_fit(weights, capacity)
print(f"Результат FF (First Fit): {ff_bins} контейнеров")

FF
Результат FF (First Fit): 178 контейнеров


### Алгоритм BF (Наилучший подходящий)  

В произвольном порядке упаковываем контейнеры по следующему правилу.  
Первый предмет помещаем в первый контейнер.  
На $k$-м шаге размещаем $k$-й контейнер. Находим частично заполненные контейнеры, где достаточно для него свободного места и выбираем среди них наиболее заполненный. Если таких нет, то берем новый пустой контейнер и помещаем предмет в него.  



In [6]:
def best_fit(weights, capacity):
    remaining_spaces = []
    
    for w in weights:
        best_index = -1
        min_remaining_space = float('inf') # ищем минимальный остаток
        
        for i in range(len(remaining_spaces)):
            # if remaining_spaces[i] >= w - 1e-9:
            if remaining_spaces[i] >= w:
                # насколько мало места останется, если положить предмет сюда?
                space_left = remaining_spaces[i] - w
                if space_left < min_remaining_space:
                    min_remaining_space = space_left
                    best_index = i
                    # перебираем контейнеры до конца
                        
        if best_index != -1:
            remaining_spaces[best_index] -= w
        # ну что поделать... достаем пустой контейнер
        else:
            remaining_spaces.append(capacity - w)
            
    return len(remaining_spaces)

In [7]:
print(f"BF")    
    
bf_bins = best_fit(weights, capacity)
print(f"Результат BF (Best Fit): {bf_bins} контейнеров")

BF
Результат BF (Best Fit): 178 контейнеров


### Offline-алгоритмы

Транспортная лента остановилась, мы окидываем взором десятки тысяч товаров и начинаем думать, как бы нам всех их расфасовать по контейнерам...

### Алгоритм FFD (Первый подходящий с упорядочиванием)  

Сортируем предметы по невозрастанию весов:  
$$w_1 \geq w_2 \geq ... \geq w_n$$
Применяем алгоритм FF.

In [8]:
def first_fit_decreasing(weights, capacity):
    sorted_weights = sorted(weights, reverse=True)
    return first_fit(sorted_weights, capacity)

In [9]:
print(f"FFD")    
    
ffd_bins = first_fit_decreasing(weights, capacity)
print(f"Результат FFD (First Fit Decreasing): {ffd_bins} контейнеров")

FFD
Результат FFD (First Fit Decreasing): 190 контейнеров


### Алгоритм NFD (Следующий подходящий с упорядочиванием)

Сортируем предметы по невозрастанию весов:  
$$w_1 \geq w_2 \geq ... \geq w_n$$
Применяем алгоритм NF.

In [10]:
def next_fit_decreasing(weights, capacity):
    sorted_weights = sorted(weights, reverse=True)
    return next_fit(sorted_weights, capacity)

In [11]:
print(f"NFD")    
    
nfd_bins = next_fit_decreasing(weights, capacity)
print(f"Результат NFD (Next Fit Decreasing): {nfd_bins} контейнеров")

NFD
Результат NFD (Next Fit Decreasing): 202 контейнеров


### Оценки

Пусть $\forall i \in I \ w_i \in [0, 1)$  
Пусть $\alpha \in [0, \frac{1}{2}]$  
Пусть 
$$\begin{cases} L_1 = \{i \in L | w_i > 1 - \alpha\} \\
L_2 = \{i \in L | \alpha \geq w_i > \frac{1}{2}\} \\
L_3 = \{i\ in L | \frac{1}{2} \geq w_i \geq \alpha\} \end{cases}$$

Тогда нижняя чут-чут оценка по весу $$H_1(\alpha) = |L_1| + |L_2| + \max \left(0, \left\lceil \sum_{i \in L_3}w_i - (|L_2| - \sum_{i \in L_2}w_i) \right\rceil\right)$$

И нижняя чут-чут оценка по количеству предметов $$H_2(\alpha) = |L_1| + |L_2| + \max \left(0, \left\lceil \frac{|L_3| - \sum_{i \in L_2} \left\lfloor \frac{1-w_i}{\alpha} \right\rfloor}{\left\lfloor\frac{1}{\alpha}\right\rfloor} \right\rceil\right)$$

Тогда настоящая нижняя оценка: $$H = \max\{H_1(\alpha), H_2(\alpha), 0 \leq \alpha \leq 0,5\}$$

In [12]:
import math

def martello_toth_lower_bound(weights, capacity):
    # нормализуем веса (как того требуют условия)
    w_norm = [w / capacity for w in weights]
    
    # начальная оценка H0 (просто сумма весов)
    H0 = math.ceil(sum(w_norm))
    H_max = H0
    
    # перебираем все возможные alpha (только те, что <= 0.5)
    # но по факту достаточно искать alpha среди весов предметов
    alphas = sorted({x for x in w_norm if x <= 0.5})
    
    for alpha in alphas:
        L1 = [x for x in w_norm if x > 1 - alpha]
        L2 = [x for x in w_norm if 1 - alpha >= x > 0.5]
        L3 = [x for x in w_norm if 0.5 >= x >= alpha]
        
        sum_L3 = sum(L3)
        free_in_L2 = len(L2) - sum(L2) 
        
        # H1
        H1 = len(L1) + len(L2) + max(0, math.ceil(sum_L3 - free_in_L2))
        
        # H2
        H2 = len(L1) + len(L2)
        if alpha > 0:
            denom = math.floor(1 / alpha)
            if denom > 0:
                sum_floor = sum(math.floor((1 - x) / alpha) for x in L2)
                H2 += max(0, math.ceil((len(L3) - sum_floor) / denom))
                
        H_max = max(H_max, H1, H2)
        
    return H_max

In [13]:
H = martello_toth_lower_bound(weights, capacity)
print(f"Нижняя оценка Martello-Toth: {H}")

Нижняя оценка Martello-Toth: 167
